# Benchmark LifeMate trên Colab

Chạy theo thứ tự từ trên xuống: **Runtime → Change runtime type → T4 GPU**, rồi chạy từng cell.

1. **Lấy code và kit.** Clone nhánh `colab-smoke` từ GitHub. Kit TRAIN lấy từ Google Drive.
2. **Cài đặt và tải model.** Qwen3.5-4B và PhoWhisper-medium, chạy offline khi benchmark.
3. **Smoke 40 mẫu.** So nhánh `direct` với `compiled` trên cùng ASR, model và 40 ID TRAIN.
4. **Tách lỗi ASR khỏi lỗi planner** (tùy chọn). Chạy lại nhánh `compiled` với transcript gốc.
5. **Lưu kết quả** lên Drive và tải file zip về để gửi phân tích.

Chưa chạy phần 269 mẫu ở cuối notebook cho tới khi đã xem kết quả smoke.

## 1. Lấy code và kit

Trước khi chạy, tải thư mục `train+public_test` lên Google Drive **một lần**. Sửa `KIT_ON_DRIVE` nếu bạn để ở chỗ khác.
Nếu repo GitHub để private, lệnh clone sẽ hỏi đăng nhập. Bạn nhập token trực tiếp trên Colab.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

BRANCH = 'colab-smoke'
REPO_URL = 'https://github.com/Quang1104/lifemate_superklang.git'
KIT_ON_DRIVE = Path('/content/drive/MyDrive/train+public_test')
RESULTS_ON_DRIVE = Path('/content/drive/MyDrive/lifemate_results')

REPO = Path('/content/lifemate_superklang')
KIT = Path('/content/train+public_test')
RUNS = REPO / 'outputs' / 'colab'
MODELS = Path('/content/lifemate_models')
PLANNER = 'planner'  # Đổi thành 'fallback_planner' rồi chạy lại từ đây nếu Qwen3.5 hết VRAM hoặc không tải được.

from google.colab import drive
drive.mount('/content/drive')

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], check=True)

if not (KIT / 'train' / 'train.jsonl').is_file():
    assert (KIT_ON_DRIVE / 'train' / 'train.jsonl').is_file(), f'Không thấy kit ở {KIT_ON_DRIVE}'
    shutil.copytree(KIT_ON_DRIVE, KIT)  # Đọc audio từ ổ local nhanh hơn đọc qua Drive.
print('Kit:', KIT)

## 2. Cài đặt và tải model

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], cwd=REPO, env=env, check=True)

run('nvidia-smi')
RUNS.mkdir(parents=True, exist_ok=True)
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[model]')
run(sys.executable, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())')

In [ ]:
from huggingface_hub import snapshot_download

pins = json.loads((REPO / 'configs' / 'colab_models.json').read_text())
for role in (PLANNER, 'asr'):
    snapshot_download(**pins[role], local_dir=MODELS / role,
                      allow_patterns=['*.json', '*.jinja', '*.safetensors', '*.bin', 'merges.txt', 'LICENSE', 'README.md'])

offline = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
catalog = KIT / 'train' / 'train_tools_catalog.json'
smoke_manifest = REPO / 'research' / 'smoke_split.json'

def benchmark(manifest, tag):
    target = RUNS / PLANNER / tag
    gold = target / 'gold.jsonl'
    run(sys.executable, '-B', '-m', 'scripts.replay_train_gold', '--data-root', KIT,
        '--manifest', manifest, '--output', gold, env=offline)
    for mode in ('direct', 'compiled'):
        run(sys.executable, '-B', '-m', 'src.inference', 'official',
            '--split', 'train', '--data-root', KIT, '--train-manifest', manifest,
            '--model-dir', MODELS / PLANNER, '--asr-dir', MODELS / 'asr', '--device', 'cuda',
            '--planner-mode', mode, '--output', target / f'{mode}.jsonl',
            '--train-report', target / f'{mode}.report.json', env=offline)
    run(sys.executable, '-B', '-m', 'scripts.compare_train', 'compare',
        '--manifest', manifest, '--gold', gold, '--catalog', catalog,
        '--direct', target / 'direct.jsonl', '--direct-report', target / 'direct.report.json',
        '--compiled', target / 'compiled.jsonl', '--compiled-report', target / 'compiled.report.json',
        '--output', target / 'comparison.json', env=offline)
    return target

## 3. Smoke 40 mẫu
Cell này in bảng tóm tắt khi chạy xong. Kết quả nằm trong `outputs/colab/<PLANNER>/smoke40/`.

In [ ]:
smoke_dir = benchmark(smoke_manifest, 'smoke40')

## 4. Tách lỗi ASR khỏi lỗi planner (tùy chọn)
Chạy lại nhánh `compiled` với transcript gốc của TRAIN, giữ nguyên model và 40 ID.
Điểm chênh giữa hai lần chạy cho biết ASR làm mất bao nhiêu điểm. Đây chỉ là phép chẩn đoán trên TRAIN.

In [ ]:
from scripts.compare_train import evaluate, index, train_ids
from research.data_audit import extract_tools, read_jsonl

run(sys.executable, '-B', '-m', 'src.inference', 'official',
    '--split', 'train', '--data-root', KIT, '--train-manifest', smoke_manifest,
    '--model-dir', MODELS / PLANNER, '--train-transcripts', '--device', 'cuda',
    '--planner-mode', 'compiled', '--output', smoke_dir / 'compiled_oracle.jsonl',
    '--train-report', smoke_dir / 'compiled_oracle.report.json', env=offline)

ids = train_ids(smoke_manifest)
gold_rows = index(read_jsonl(smoke_dir / 'gold.jsonl'))
gold = {sample_id: gold_rows[sample_id] for sample_id in ids}
tools = extract_tools(json.loads(catalog.read_text(encoding='utf-8-sig')))
asr_scores, asr = evaluate(gold, smoke_dir / 'compiled.jsonl', smoke_dir / 'compiled.report.json', tools)
oracle_scores, oracle = evaluate(gold, smoke_dir / 'compiled_oracle.jsonl', smoke_dir / 'compiled_oracle.report.json', tools)
asr_vs_oracle = {
    'compiled_asr_proxy': asr['local_proxy_0_to_100'],
    'compiled_oracle_proxy': oracle['local_proxy_0_to_100'],
    'points_lost_to_asr': oracle['local_proxy_0_to_100'] - asr['local_proxy_0_to_100'],
    'oracle_losses_by_outcome': oracle['lost_proxy_points_by_outcome'],
    'samples_hurt_by_asr': [i for i in ids if asr_scores[i] < oracle_scores[i]],
}
(smoke_dir / 'asr_vs_oracle.json').write_text(json.dumps(asr_vs_oracle, ensure_ascii=False, indent=2))
print(json.dumps(asr_vs_oracle, ensure_ascii=False, indent=2))

## 5. Lưu và tải kết quả
Chép thư mục kết quả lên Drive và tải một file zip về máy. Gửi file zip này để phân tích.

In [ ]:
from google.colab import files

stamp = subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
                       capture_output=True, text=True, check=True).stdout.strip()
name = f'{PLANNER}_smoke40_{stamp}'
archive = shutil.make_archive(f'/content/{name}', 'zip', smoke_dir)
RESULTS_ON_DRIVE.mkdir(parents=True, exist_ok=True)
shutil.copy(archive, RESULTS_ON_DRIVE)
print('Đã lưu lên Drive:', RESULTS_ON_DRIVE / Path(archive).name)
files.download(archive)

## 269 mẫu ngoài smoke
Chỉ chạy sau khi đã chốt checkpoint, prompt và thuật toán từ smoke. Không chỉnh cấu hình theo kết quả của phần này.

In [ ]:
locked_manifest = RUNS / 'locked269.json'
run(sys.executable, '-B', '-m', 'scripts.compare_train', 'prepare', '--output', locked_manifest)
locked_dir = benchmark(locked_manifest, 'locked269')
archive = shutil.make_archive(f'/content/{PLANNER}_locked269', 'zip', locked_dir)
RESULTS_ON_DRIVE.mkdir(parents=True, exist_ok=True)
shutil.copy(archive, RESULTS_ON_DRIVE)